In [1]:

# Project: Olist E-Commerce Analytics
# Purpose: Evaluate low-rating classification on unseen orders.

import pandas as pd
import numpy as np
import pyodbc

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    confusion_matrix
)

# Connect to our existing SQL Server analytical view.
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=Sindhu;"
    "DATABASE=OlistEcommerceAnalytics;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

query = """
SELECT
    order_id,
    order_purchase_timestamp,
    customer_state,
    is_late,
    low_rating_flag,
    item_count,
    seller_count,
    order_item_value_brl,
    order_freight_brl,
    promised_delivery_days
FROM analytics.vw_delivery_satisfaction_features
"""

# Fetch SQL records and construct the modeling DataFrame.
with pyodbc.connect(connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    columns = [column[0] for column in cursor.description]
    rows = [tuple(row) for row in cursor.fetchall()]

df = pd.DataFrame(rows, columns=columns)

# Convert purchase timestamps and required numeric attributes.
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

numeric_features = [
    "is_late",
    "item_count",
    "seller_count",
    "order_item_value_brl",
    "order_freight_brl",
    "promised_delivery_days"
]

for column in numeric_features + ["low_rating_flag"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Represent purchase month as a categorical variable.
df["purchase_month"] = (
    df["order_purchase_timestamp"].dt.month.astype(str)
)

categorical_features = [
    "customer_state",
    "purchase_month"
]

# Exclude rows lacking required modeling inputs.
required = (
    numeric_features
    + categorical_features
    + ["low_rating_flag", "order_purchase_timestamp"]
)

df = df.dropna(subset=required).copy()

# Sort chronologically to avoid random future-to-past splitting.
df = df.sort_values(
    ["order_purchase_timestamp", "order_id"]
).reset_index(drop=True)

# Use the earliest 80% for training and latest 20% for testing.
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

y_train = train_df["low_rating_flag"].astype(int)
y_test = test_df["low_rating_flag"].astype(int)

# Standardize numeric values and encode categorical attributes.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Build a regularized logistic regression classification pipeline.
classifier = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Train only on earlier historical orders.
classifier.fit(X_train, y_train)

# Predict low-rating probabilities for held-out orders.
y_probability = classifier.predict_proba(X_test)[:, 1]

# Apply an initial 0.50 decision threshold.
y_predicted = (y_probability >= 0.50).astype(int)

# Evaluate discrimination, precision, recall and calibration.
evaluation = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Precision",
        "Recall",
        "Brier Score",
        "Test Low-Rating Prevalence"
    ],
    "Value": [
        roc_auc_score(y_test, y_probability),
        average_precision_score(y_test, y_probability),
        precision_score(y_test, y_predicted, zero_division=0),
        recall_score(y_test, y_predicted, zero_division=0),
        brier_score_loss(y_test, y_probability),
        y_test.mean()
    ]
})

# Print the time boundaries and sample sizes.
print("TRAINING AND TESTING SUMMARY")
print("Training orders:", len(train_df))
print("Testing orders:", len(test_df))
print("Training ends:", train_df["order_purchase_timestamp"].max())
print("Testing starts:", test_df["order_purchase_timestamp"].min())

# Display all classification evaluation metrics.
print("\nOUT-OF-SAMPLE MODEL PERFORMANCE")
display(evaluation.round(4))

# Display the confusion matrix at a 0.50 threshold.
cm = confusion_matrix(y_test, y_predicted)

print("\nCONFUSION MATRIX")
display(pd.DataFrame(
    cm,
    index=["Actual Other", "Actual Low"],
    columns=["Predicted Other", "Predicted Low"]
))


TRAINING AND TESTING SUMMARY
Training orders: 76659
Testing orders: 19165
Training ends: 2018-05-27 11:13:44
Testing starts: 2018-05-27 11:41:32

OUT-OF-SAMPLE MODEL PERFORMANCE


,Metric,Value
0,ROC-AUC,0.6631
1,PR-AUC,0.2663
2,Precision,0.5208
3,Recall,0.1419
4,Brier Score,0.0799
5,Test Low-Rating Prevalence,0.0967



CONFUSION MATRIX


,Predicted Other,Predicted Low
Actual Other,17069,242
Actual Low,1591,263


In [2]:
from sklearn.base import clone
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score
)
from IPython.display import display

# Define chronological training and validation boundaries.
train_end = int(len(df) * 0.64)
validation_end = int(len(df) * 0.80)

# Keep the final 20% separate from threshold selection.
training_data = df.iloc[:train_end].copy()
validation_data = df.iloc[
    train_end:validation_end
].copy()

# Separate explanatory features and target labels.
X_training = training_data[feature_columns]
y_training = training_data["low_rating_flag"].astype(int)

X_validation = validation_data[feature_columns]
y_validation = validation_data["low_rating_flag"].astype(int)

# Clone the existing pipeline to create a fresh model.
# This avoids using information from the validation period
# when fitting preprocessing steps and coefficients.
validation_model = clone(classifier)

# Train using only the earliest historical observations.
validation_model.fit(X_training, y_training)

# Predict probabilities on the later validation period.
validation_probabilities = (
    validation_model.predict_proba(X_validation)[:, 1]
)

# Calculate precision and recall at available thresholds.
precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_validation,
        validation_probabilities
    )
)

# Display performance at several practical thresholds.
candidate_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]

threshold_results = []

for threshold in candidate_thresholds:

    # Classify orders using the candidate threshold.
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    # Calculate classification performance.
    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Orders Flagged": int(predictions.sum())
    })

# Display the threshold comparison.
threshold_df = pd.DataFrame(threshold_results)

print("VALIDATION THRESHOLD COMPARISON")
display(threshold_df.round(4))

# Define the minimum recall needed for our business goal.
target_recall = 0.50

# Exclude the final precision/recall endpoint because
# it has no corresponding classification threshold.
eligible_indices = np.where(
    recall_values[:-1] >= target_recall
)[0]

# Choose the eligible threshold with highest precision.
if len(eligible_indices) > 0:

    best_index = eligible_indices[
        np.argmax(precision_values[eligible_indices])
    ]

    best_threshold = float(thresholds[best_index])

    print("\nVALIDATION-SELECTED THRESHOLD")
    print("Selected threshold:", round(best_threshold, 4))
    print(
        "Validation precision:",
        round(float(precision_values[best_index]), 4)
    )
    print(
        "Validation recall:",
        round(float(recall_values[best_index]), 4)
    )

else:
    # Handle an unexpected inability to meet the target.
    best_threshold = None
    print("No threshold met the recall target.")

# Confirm that the final test period was not used
# to choose the threshold.
print("\nTraining records:", len(training_data))
print("Validation records:", len(validation_data))
print("Test records reserved:",
      len(df) - validation_end)


VALIDATION THRESHOLD COMPARISON


,Threshold,Precision,Recall,F1 Score,Orders Flagged
0,0.1,0.2813,0.5710,0.3769,3789
1,0.2,0.5064,0.4017,0.4480,1481
2,0.3,0.5259,0.3862,0.4453,1371
3,0.4,0.5421,0.3589,0.4318,1236
4,0.5,0.5549,0.3278,0.4121,1103



VALIDATION-SELECTED THRESHOLD
Selected threshold: 0.1174
Validation precision: 0.3673
Validation recall: 0.5003

Training records: 61327
Validation records: 15332
Test records reserved: 19165


In [3]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from IPython.display import display

# Use the probabilities from the model trained on
# the earliest 80% of orders in our earlier notebook cell.
# Do not select or tune thresholds using test labels.
test_probabilities = y_probability

# Use the threshold selected on validation data.
selected_threshold = best_threshold

# Confirm the test dataset and predictions align.
assert selected_threshold is not None
assert len(test_probabilities) == len(y_test)

# Compare the default and validation-selected thresholds.
thresholds_to_compare = {
    "Default (0.50)": 0.50,
    "Validation-Selected": selected_threshold
}

comparison_results = []

for name, threshold in thresholds_to_compare.items():

    # Convert predicted probabilities into binary classes.
    predictions = (
        test_probabilities >= threshold
    ).astype(int)

    # Calculate precision, recall and F1 score.
    comparison_results.append({
        "Threshold Strategy": name,
        "Threshold": round(threshold, 4),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "Orders Flagged": int(predictions.sum())
    })

# Display the out-of-sample threshold comparison.
comparison_df = pd.DataFrame(comparison_results)

print("TEST SET — THRESHOLD COMPARISON")
display(comparison_df.round(4))

# Evaluate the validation-selected threshold in detail.
final_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

# Create the confusion matrix.
cm = confusion_matrix(
    y_test,
    final_predictions,
    labels=[0, 1]
)

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Other", "Actual Low"],
    columns=["Predicted Other", "Predicted Low"]
)

print("\nSELECTED THRESHOLD — CONFUSION MATRIX")
display(confusion_df)

# Report how many negative reviews were identified
# and how many were missed at the selected threshold.
tn, fp, fn, tp = cm.ravel()

print("\nBUSINESS IMPACT SUMMARY")
print("Correctly flagged low ratings:", tp)
print("Missed low ratings:", fn)
print("False alerts:", fp)
print("Correctly ignored other ratings:", tn)
print("Total orders flagged:", tp + fp)


TEST SET — THRESHOLD COMPARISON


,Threshold Strategy,Threshold,Precision,Recall,F1 Score,Orders Flagged
0,Default (0.50),0.5000,0.5208,0.1419,0.2230,505
1,Validation-Selected,0.1174,0.2861,0.3689,0.3223,2391



SELECTED THRESHOLD — CONFUSION MATRIX


,Predicted Other,Predicted Low
Actual Other,15604,1707
Actual Low,1170,684



BUSINESS IMPACT SUMMARY
Correctly flagged low ratings: 684
Missed low ratings: 1170
False alerts: 1707
Correctly ignored other ratings: 15604
Total orders flagged: 2391


In [4]:
# Use the model trained on the earliest 64% of orders.
# This model was also used to select our threshold
# on the subsequent 16% validation period.
consistent_model = validation_model

# Use the previously selected validation threshold.
threshold = best_threshold

# Verify that the threshold is available.
assert threshold is not None

# Generate test probabilities using the SAME model.
# Do not refit the model using test data.
consistent_test_probabilities = (
    consistent_model.predict_proba(X_test)[:, 1]
)

# Classify held-out orders using the fixed threshold.
consistent_predictions = (
    consistent_test_probabilities >= threshold
).astype(int)

# Calculate classification performance.
results = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "Average Precision",
        "Precision",
        "Recall",
        "F1 Score",
        "Selected Threshold",
        "Orders Flagged"
    ],
    "Value": [
        roc_auc_score(
            y_test, consistent_test_probabilities
        ),
        average_precision_score(
            y_test, consistent_test_probabilities
        ),
        precision_score(
            y_test, consistent_predictions,
            zero_division=0
        ),
        recall_score(
            y_test, consistent_predictions,
            zero_division=0
        ),
        f1_score(
            y_test, consistent_predictions,
            zero_division=0
        ),
        threshold,
        int(consistent_predictions.sum())
    ]
})

# Display corrected test evaluation results.
print("CONSISTENT-MODEL TEST EVALUATION")
display(results.round(4))

# Calculate the confusion matrix for the selected threshold.
cm = confusion_matrix(
    y_test,
    consistent_predictions,
    labels=[0, 1]
)

# Display correctly and incorrectly classified orders.
print("\nCONFUSION MATRIX")
display(
    pd.DataFrame(
        cm,
        index=["Actual Other", "Actual Low"],
        columns=["Predicted Other", "Predicted Low"]
    )
)

# Report the original validation target and test outcome.
test_recall = recall_score(
    y_test, consistent_predictions,
    zero_division=0
)

print("\nRECALL TARGET CHECK")
print("Target recall: 50%")
print(f"Observed test recall: {test_recall * 100:.2f}%")
print("Target achieved:", test_recall >= 0.50)


CONSISTENT-MODEL TEST EVALUATION


,Metric,Value
0,ROC-AUC,0.6633
1,Average Precision,0.2660
2,Precision,0.2855
3,Recall,0.3706
4,F1 Score,0.3225
5,Selected Threshold,0.1174
6,Orders Flagged,2406.0000



CONFUSION MATRIX


,Predicted Other,Predicted Low
Actual Other,15592,1719
Actual Low,1167,687



RECALL TARGET CHECK
Target recall: 50%
Observed test recall: 37.06%
Target achieved: False


In [5]:
# Reuse the chronological split boundaries from
# the earlier model validation workflow.
train_end = int(len(df) * 0.64)
validation_end = int(len(df) * 0.80)

# Separate the historical observations into three periods.
datasets = {
    "Training": df.iloc[:train_end].copy(),
    "Validation": df.iloc[train_end:validation_end].copy(),
    "Testing": df.iloc[validation_end:].copy()
}

# Store descriptive statistics for each period.
drift_results = []

for period_name, data in datasets.items():

    # Identify late deliveries and low-rated orders.
    late_mask = data["is_late"] == 1
    on_time_mask = data["is_late"] == 0

    # Calculate overall negative-review prevalence.
    low_rating_rate = (
        data["low_rating_flag"].mean() * 100
    )

    # Calculate the percentage of orders delivered late.
    late_delivery_rate = (
        late_mask.mean() * 100
    )

    # Calculate negative-review rates within delivery groups.
    on_time_low_rating_rate = (
        data.loc[
            on_time_mask, "low_rating_flag"
        ].mean() * 100
    )

    late_low_rating_rate = (
        data.loc[
            late_mask, "low_rating_flag"
        ].mean() * 100
    )

    # Collect the period-level distribution statistics.
    drift_results.append({
        "Period": period_name,
        "Orders": len(data),
        "Start Date": data[
            "order_purchase_timestamp"
        ].min().date(),
        "End Date": data[
            "order_purchase_timestamp"
        ].max().date(),
        "Low Rating %": round(low_rating_rate, 2),
        "Late Delivery %": round(late_delivery_rate, 2),
        "On-Time Low Rating %": round(
            on_time_low_rating_rate, 2
        ),
        "Late Low Rating %": round(
            late_low_rating_rate, 2
        )
    })

# Display the distribution comparison across periods.
drift_df = pd.DataFrame(drift_results)

print("TRAINING VS VALIDATION VS TEST DISTRIBUTION")
display(drift_df)

# Compare validation and test low-rating prevalence.
validation_rate = drift_df.loc[
    drift_df["Period"] == "Validation",
    "Low Rating %"
].iloc[0]

test_rate = drift_df.loc[
    drift_df["Period"] == "Testing",
    "Low Rating %"
].iloc[0]

# Report the observed difference in percentage points.
print(
    "\nChange in low-rating prevalence (test - validation):",
    round(test_rate - validation_rate, 2),
    "percentage points"
)


TRAINING VS VALIDATION VS TEST DISTRIBUTION


,Period,Orders,Start Date,End Date,Low Rating %,Late Delivery %,On-Time Low Rating %,Late Low Rating %
0,Training,61327,2016-09-15,2018-03-20,13.94,7.70,9.61,65.92
1,Validation,15332,2018-03-20,2018-05-27,12.18,6.54,9.10,56.13
2,Testing,19165,2018-05-27,2018-08-29,9.67,3.43,8.35,46.88



Change in low-rating prevalence (test - validation): -2.51 percentage points


In [6]:
# Reuse the model selected on the training/validation data.
# The test probabilities come from the same fitted model.
selected_threshold = best_threshold

# Define the validation and testing datasets with
# their corresponding predicted probabilities.
periods = [
    (
        "Validation",
        validation_data,
        validation_probabilities
    ),
    (
        "Testing",
        test_df,
        consistent_test_probabilities
    )
]

# Store performance metrics for each delivery group.
subgroup_results = []

for period_name, data, probabilities in periods:

    # Create a temporary copy to avoid altering
    # the original training and testing datasets.
    diagnostics = data[
        ["is_late", "low_rating_flag"]
    ].copy().reset_index(drop=True)

    # Attach the probabilities produced by the same model.
    assert len(diagnostics) == len(probabilities)

    diagnostics["predicted_probability"] = np.asarray(
        probabilities
    )

    # Apply the fixed validation-selected threshold.
    diagnostics["predicted_low_rating"] = (
        diagnostics["predicted_probability"]
        >= selected_threshold
    ).astype(int)

    # Analyze on-time and late orders independently.
    for delivery_flag, delivery_name in [
        (0, "On-Time"),
        (1, "Late")
    ]:

        # Filter the current delivery subgroup.
        group = diagnostics[
            diagnostics["is_late"] == delivery_flag
        ]

        # Count actual low ratings and model predictions.
        actual_low = int(group["low_rating_flag"].sum())

        flagged = int(
            group["predicted_low_rating"].sum()
        )

        true_positives = int(
            (
                (group["low_rating_flag"] == 1)
                & (group["predicted_low_rating"] == 1)
            ).sum()
        )

        # Calculate subgroup precision and recall.
        precision = (
            true_positives / flagged
            if flagged > 0 else 0
        )

        recall = (
            true_positives / actual_low
            if actual_low > 0 else 0
        )

        # Save the findings for side-by-side comparison.
        subgroup_results.append({
            "Period": period_name,
            "Delivery Status": delivery_name,
            "Orders": len(group),
            "Actual Low Ratings": actual_low,
            "Correctly Flagged": true_positives,
            "Missed Low Ratings": (
                actual_low - true_positives
            ),
            "Precision %": round(
                precision * 100, 2
            ),
            "Recall %": round(
                recall * 100, 2
            ),
            "Average Predicted Risk %": round(
                group["predicted_probability"].mean()
                * 100, 2
            )
        })

# Display subgroup model-performance results.
subgroup_df = pd.DataFrame(subgroup_results)

print("MODEL PERFORMANCE BY DELIVERY STATUS")
display(subgroup_df)


MODEL PERFORMANCE BY DELIVERY STATUS


,Period,Delivery Status,Orders,Actual Low Ratings,Correctly Flagged,Missed Low Ratings,Precision %,Recall %,Average Predicted Risk %
0,Validation,On-Time,14329,1304,371,933,24.09,28.45,9.40
1,Validation,Late,1003,563,563,0,56.13,100.00,62.94
2,Testing,On-Time,18508,1546,379,1167,21.67,24.51,8.65
3,Testing,Late,657,308,308,0,46.88,100.00,55.50


In [7]:

# Reuse the test predictions from our consistent model.
# Preserve row order so each probability matches its order.
diagnostics = test_df[
    ["order_id", "is_late", "low_rating_flag"]
].copy().reset_index(drop=True)

# Attach predictions without retraining or changing thresholds.
assert len(diagnostics) == len(consistent_test_probabilities)

diagnostics["predicted_probability"] = np.asarray(
    consistent_test_probabilities
)

# Select only on-time deliveries from the historical test set.
on_time_test = diagnostics[
    diagnostics["is_late"] == 0
].copy()

# Calculate observed negative-rating prevalence.
baseline_rate = on_time_test["low_rating_flag"].mean()

# Measure discrimination and ranking quality.
on_time_auc = roc_auc_score(
    on_time_test["low_rating_flag"],
    on_time_test["predicted_probability"]
)

on_time_ap = average_precision_score(
    on_time_test["low_rating_flag"],
    on_time_test["predicted_probability"]
)

# Sort on-time orders from highest to lowest model risk.
on_time_ranked = on_time_test.sort_values(
    ["predicted_probability", "order_id"],
    ascending=[False, True]
)

# Examine the highest-risk 10% of on-time orders.
top_count = int(np.ceil(len(on_time_ranked) * 0.10))
top_risk = on_time_ranked.head(top_count)

# Calculate precision among the highest-risk orders.
top_precision = top_risk["low_rating_flag"].mean()

# Calculate the fraction of low ratings captured
# by the top 10% highest-risk group.
top_recall = (
    top_risk["low_rating_flag"].sum()
    / on_time_ranked["low_rating_flag"].sum()
)

# Compare high-risk-group precision with overall prevalence.
lift = top_precision / baseline_rate

# Display the diagnostic results.
results = pd.DataFrame({
    "Metric": [
        "On-Time Test Orders",
        "Actual Low-Rating Orders",
        "Baseline Low-Rating Rate (%)",
        "On-Time ROC-AUC",
        "On-Time Average Precision",
        "Top 10% Orders Examined",
        "Top 10% Low-Rating Precision (%)",
        "Top 10% Low-Rating Recall (%)",
        "Top 10% Lift"
    ],
    "Value": [
        len(on_time_test),
        int(on_time_test["low_rating_flag"].sum()),
        round(baseline_rate * 100, 2),
        round(on_time_auc, 4),
        round(on_time_ap, 4),
        top_count,
        round(top_precision * 100, 2),
        round(top_recall * 100, 2),
        round(lift, 2)
    ]
})

print("ON-TIME CUSTOMER DISSATISFACTION ANALYSIS")
display(results)


ON-TIME CUSTOMER DISSATISFACTION ANALYSIS


,Metric,Value
0,On-Time Test Orders,18508.0000
1,Actual Low-Rating Orders,1546.0000
2,Baseline Low-Rating Rate (%),8.3500
3,On-Time ROC-AUC,0.6100
4,On-Time Average Precision,0.1719
5,Top 10% Orders Examined,1851.0000
6,Top 10% Low-Rating Precision (%),21.1200
7,Top 10% Low-Rating Recall (%),25.2900
8,Top 10% Lift,2.5300
